 Notebook: erp_ingest_inventory

 Layer: Ingestion → Bronze

 Source: ERP (CSV files from Unity Catalog Volume)

 Description:
 - Validates raw data (batch - commented)
 - Ingests data using Auto Loader (incremental)
 - Writes to Bronze Delta table

 Architecture:
 Volumes → Auto Loader → Bronze




In [0]:
source_path="/Volumes/workspace/default/raw_data/erp/inventory/"


Read raw CSV files from volume
- Reads all files in folder
- Uses correct delimiter (,)
- No transformations (raw ingestion)


In [0]:
df_inventory=spark.read\
    .format("csv")\
    .option("header",True)\
    .option("inferschema",True)\
    .load(source_path)


 Validate data



In [0]:
df_inventory.printSchema()
display(df_inventory)


  Row count check


In [0]:
df_inventory.count()


Step 3: Define source and checkpoint paths


In [0]:
source_path="/Volumes/workspace/default/raw_data/erp/inventory/"
checkpoint_path="/Volumes/workspace/default/raw_data/checkpoints/inventory_checkpoint"
schema_location = "/Volumes/workspace/default/raw_data/schemas/inventory"

 Step 4: Define Schema


In [0]:
import pyspark.sql.types as T

inventory_schema=T.StructType([
    T.StructField("product_id",T.StringType(),True),
    T.StructField("warehouse_id",T.StringType(),True),
    T.StructField("stock_quantity",T.DoubleType(),True),
    T.StructField("reorder_level",T.IntegerType(),True),
    T.StructField("last_updated",T.StringType(),True),
])


 Step 5: Read inventory using Auto Loader (with metadata)
 - Incremental ingestion
 - Detects new files automatically
 - Adds ingestion metadata (timestamp + source file)
 - Handles schema evolution


In [0]:
import pyspark.sql.functions as F

df_inventory_stream = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)   
    .option("header", True)
    .option("delimiter", ",")
    .schema(inventory_schema)
    .load(source_path)
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)


 Step 6: Write stream to Bronze Delta table
 - Stores raw ERP inventory
 - Append-only (no transformations)
 - Checkpoint ensures incremental processing

In [0]:
(
    df_inventory_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("bronze.erp_inventory")
)


 Step 7: Validate Bronze table
 - Verify ingestion worked
 - Check metadata columns


In [0]:
display(spark.table("bronze.erp_inventory"))

In [0]:
%sql
Select count(*)
from bronze.erp_inventory